In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = ("https://raw.githubusercontent.com/jxchen/Kaggle"
       "/master/Give%20Me%20Some%20Credit/cs-training.csv")

df_raw = pd.read_csv(url, index_col=0)
# index_col=0 → cột đầu tiên làm index (số thứ tự khách hàng)
# không có index_col → Pandas tự tạo 0,1,2,... → bị thừa cột

print(f"Shape: {df_raw.shape}")
# → (150000, 11): 150,000 khách hàng, 11 chỉ số tín dụng

Shape: (150000, 11)


In [2]:
df = df_raw.rename(columns={
    "SeriousDlqin2yrs":                    "vo_no",
    "RevolvingUtilizationOfUnsecuredLines": "ty_le_su_dung_tin_dung",
    "age":                                  "tuoi",
    "NumberOfTime30-59DaysPastDueNotWorse": "so_lan_tre_30_59_ngay",
    "DebtRatio":                            "ty_le_no",
    "MonthlyIncome":                        "thu_nhap_thang",
    "NumberOfOpenCreditLinesAndLoans":      "so_tai_khoan_vay",
    "NumberOfTimes90DaysLate":              "so_lan_tre_90_ngay",
    "NumberRealEstateLoansOrLines":         "so_tai_khoan_bat_dong_san",
    "NumberOfTime60-89DaysPastDueNotWorse": "so_lan_tre_60_89_ngay",
    "NumberOfDependents":                   "so_nguoi_phu_thuoc"
})

print(f"Shape: {df.shape} | Tổng NaN: {df.isnull().sum().sum()}")

Shape: (150000, 11) | Tổng NaN: 33655


In [3]:
# Kiểu dữ liệu từng cột — có đúng không?
print(df.dtypes)
# Kỳ vọng: vo_no → int, tuoi → int, thu_nhap_thang → float

# Thông tin tổng quan — cột nào có NaN?
df.info()
# "Non-Null Count" < 150000 → có NaN

# Thống kê mô tả nhanh
df.describe()
# Nhìn vào: min, max, mean của từng cột
# Có gì bất thường không? (tuoi=0? thu nhập âm?)

vo_no                          int64
ty_le_su_dung_tin_dung       float64
tuoi                           int64
so_lan_tre_30_59_ngay          int64
ty_le_no                     float64
thu_nhap_thang               float64
so_tai_khoan_vay               int64
so_lan_tre_90_ngay             int64
so_tai_khoan_bat_dong_san      int64
so_lan_tre_60_89_ngay          int64
so_nguoi_phu_thuoc           float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
Index: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   vo_no                      150000 non-null  int64  
 1   ty_le_su_dung_tin_dung     150000 non-null  float64
 2   tuoi                       150000 non-null  int64  
 3   so_lan_tre_30_59_ngay      150000 non-null  int64  
 4   ty_le_no                   150000 non-null  float64
 5   thu_nhap_thang             120269 non-null  float64
 6   so_tai_khoan

,vo_no,ty_le_su_dung_tin_dung,tuoi,so_lan_tre_30_59_ngay,ty_le_no,thu_nhap_thang,so_tai_khoan_vay,so_lan_tre_90_ngay,so_tai_khoan_bat_dong_san,so_lan_tre_60_89_ngay,so_nguoi_phu_thuoc
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [4]:
# 1 ngoặc vuông → Series (1 chiều)
s = df["thu_nhap_thang"]
print(type(s))   # <class 'pandas.core.series.Series'>
print(s.shape)   # (150000,) — chỉ có 1 chiều

# 2 ngoặc vuông → DataFrame (2 chiều)
df2 = df[["tuoi", "thu_nhap_thang"]]
print(type(df2))  # <class 'pandas.core.frame.DataFrame'>
print(df2.shape)  # (150000, 2)

# Tại sao quan trọng?
# Nhiều hàm Pandas chỉ nhận DataFrame, không nhận Series
# → biết cái nào mình đang cầm là kỹ năng cơ bản

<class 'pandas.core.series.Series'>
(150000,)
<class 'pandas.core.frame.DataFrame'>
(150000, 2)


In [5]:
# SAI — Pandas 2.x sẽ cảnh báo hoặc không thay đổi df gốc:
df_senior = df[df["tuoi"] > 60]
df_senior["nhom_tuoi"] = "Senior"  # ⚠️ SettingWithCopyWarning

# ĐÚNG — phải .copy() khi filter rồi sửa:
df_senior = df[df["tuoi"] > 60].copy()   # tạo bản sao độc lập
df_senior.loc[:, "nhom_tuoi"] = "Senior" # OK — không warning
print(df_senior["nhom_tuoi"].unique())    # ['Senior']

['Senior']


/tmp/ipykernel_4771/2964951401.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_senior["nhom_tuoi"] = "Senior"  # ⚠️ SettingWithCopyWarning


In [6]:
# Tính tỷ lệ vỡ nợ
ty_le_vo_no = df["vo_no"].mean()
so_vo_no    = df["vo_no"].sum()
so_khong    = len(df) - so_vo_no

print(f"Tổng khách hàng : {len(df):,}")
print(f"Vỡ nợ           : {so_vo_no:,}")
print(f"Không vỡ nợ     : {so_khong:,}")
print(f"Tỷ lệ vỡ nợ     : {ty_le_vo_no:.1%}")

Tổng khách hàng : 150,000
Vỡ nợ           : 10,026
Không vỡ nợ     : 139,974
Tỷ lệ vỡ nợ     : 6.7%


In [7]:
nan_info = df.isnull().sum()
nan_pct  = df.isnull().mean() * 100

nan_report = pd.DataFrame({
    "so_nan"   : nan_info,
    "phan_tram": nan_pct.round(2)
})

print(nan_report[nan_report["so_nan"] > 0])

                    so_nan  phan_tram
thu_nhap_thang       29731      19.82
so_nguoi_phu_thuoc    3924       2.62


In [8]:
a = df["vo_no"]
b = df[["vo_no"]]

print(type(a), a.shape)
print(type(b), b.shape)

print("--- a.mean() ---")
print(a.mean())

print("--- b.mean() ---")
print(b.mean())

<class 'pandas.core.series.Series'> (150000,)
<class 'pandas.core.frame.DataFrame'> (150000, 1)
--- a.mean() ---
0.06684
--- b.mean() ---
vo_no    0.06684
dtype: float64
